# RF-DETR Nano for NSFW object detection — with Hyperparameter Tuning

Optuna **RandomSearch** over the RF-DETR training hyperparameters (base learning
rate, encoder LR multiplier, weight decay, backbone layer-wise LR decay,
stochastic-depth `drop_path`, and batch size), followed by a full-length retrain
with the best configuration. Mirrors the structure of the EfficientDet /
SSD-MobileNet / NLP `_hyperparameters` notebooks.

The base pipeline matches `rf_detr_nano.ipynb`: `RFDETRNano` (DINOv2-windowed
backbone, COCO-pretrained) fine-tuned on the Roboflow `erotica-detection` v4
dataset in **COCO** format at **640×640**.

Because RF-DETR trains through its own high-level `model.train()` API (not a
hand-written loop), each trial launches a fresh model, trains for a few short
`SEARCH_EPOCHS`, and is scored by the **best validation mAP@[.5:.95]** parsed
from RF-DETR's `log.txt`. The best config is then retrained for the full
`FINAL_EPOCHS`.

> Heads-up: HPO here is expensive — every trial is a real (short) RF-DETR
> training run. Lower `MAX_TRIALS`, `SEARCH_EPOCHS`, or `SEARCH_RESOLUTION` if
> you are time/GPU constrained.

In [ ]:
import os
from google.colab import userdata

os.environ["ROBOFLOW_API_KEY"] = userdata.get("ROBOFLOW_API_KEY")

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q rfdetr==1.2.1 supervision==0.26.1 roboflow optuna

In [ ]:
from roboflow import Roboflow
from google.colab import userdata

ROBOFLOW_API_KEY = userdata.get('ROBOFLOW_API_KEY')
rf = Roboflow(api_key=ROBOFLOW_API_KEY)

workspace = rf.workspace("renaire-odarve")
project = workspace.project("erotica-detection")
version = project.version(4)  # same dataset/version as the other CV notebooks
dataset = version.download("coco")  # RF-DETR requires COCO format
DATASET_DIR = dataset.location
print("Dataset at:", DATASET_DIR)

# HYPERPARAMETER OPTIMIZATION (Optuna RandomSearch)

Each trial trains a fresh `RFDETRNano` for a few `SEARCH_EPOCHS` and is scored by
the **best validation mAP@[.5:.95]** it reaches (parsed from `log.txt`). Trial
checkpoints are not saved (`dont_save_weights=True`) to keep disk usage low —
only the eval metrics are needed. The best configuration is then retrained in
full.

In [ ]:
# ============================================================
# Config & helpers (shared by HPO search and final training)
# ============================================================
import os
import gc
import json
import random
import numpy as np
import torch
from rfdetr import RFDETRNano

SEED = 42

# --- HPO controls (RandomSearch, mirrors the other _hyperparameters notebooks) ---
MAX_TRIALS        = 10    # number of Optuna trials
SEARCH_EPOCHS     = 5     # short epochs per trial (detectors need a couple to show mAP signal)
SEARCH_RESOLUTION = 640   # lower (e.g. 448/384) to speed the search; keep divisible by 64

# --- Final training (after HPO) ---
FINAL_EPOCHS      = 40
FINAL_RESOLUTION  = 640   # must match the reload/eval resolution below

HPO_DIR   = "/content/hpo"
FINAL_DIR = "/content/output"
os.makedirs(HPO_DIR, exist_ok=True)


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def cleanup_gpu_memory(obj=None):
    """Drop references to a model and reclaim GPU memory between trials."""
    if obj is not None:
        del obj
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


def best_map_from_log(output_dir):
    """Parse RF-DETR's log.txt and return the best COCO mAP@[.5:.95].

    RF-DETR writes one JSON object per epoch; the COCO eval vector is stored
    under 'test_coco_eval_bbox' and its first element is AP@[IoU=0.50:0.95].
    """
    log_path = os.path.join(output_dir, "log.txt")
    best = 0.0
    if not os.path.exists(log_path):
        return best
    with open(log_path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                entry = json.loads(line)
            except ValueError:
                continue
            coco = entry.get("test_coco_eval_bbox")
            if isinstance(coco, list) and coco:
                best = max(best, float(coco[0]))
    return best

In [ ]:
# ============================================================
# Hyperparameter search space + Optuna RandomSearch objective
# ============================================================
import optuna


def rfdetr_hp_space(trial):
    """Sample one RF-DETR training configuration.

    Mirrors the tuning scale of the other _hyperparameters notebooks (lr,
    weight_decay, batch_size) plus RF-DETR-specific knobs: the encoder LR
    multiplier, the ViT layer-wise LR decay, and stochastic-depth drop_path.
    """
    lr = trial.suggest_float("lr", 5e-5, 5e-4, log=True)
    return {
        "lr": lr,
        "lr_encoder_mult":    trial.suggest_float("lr_encoder_mult", 1.0, 2.0, step=0.25),
        "weight_decay":       trial.suggest_float("weight_decay", 1e-5, 1e-2, log=True),
        "lr_vit_layer_decay": trial.suggest_float("lr_vit_layer_decay", 0.70, 0.90, step=0.05),
        "drop_path":          trial.suggest_float("drop_path", 0.0, 0.20, step=0.05),
        "batch_size":         trial.suggest_categorical("batch_size", [16, 32]),
    }


def objective(trial):
    hp = rfdetr_hp_space(trial)
    trial_dir = os.path.join(HPO_DIR, f"trial_{trial.number}")

    model = RFDETRNano()
    score = 0.0
    try:
        model.train(
            dataset_dir=DATASET_DIR,
            output_dir=trial_dir,
            epochs=SEARCH_EPOCHS,
            batch_size=hp["batch_size"],
            grad_accum_steps=1,
            resolution=SEARCH_RESOLUTION,
            lr=hp["lr"],
            lr_encoder=hp["lr"] * hp["lr_encoder_mult"],
            weight_decay=hp["weight_decay"],
            lr_vit_layer_decay=hp["lr_vit_layer_decay"],
            drop_path=hp["drop_path"],
            dont_save_weights=True,   # search only needs eval metrics -> saves disk
            tensorboard=False,
        )
        score = best_map_from_log(trial_dir)
    finally:
        cleanup_gpu_memory(model)

    print(f"[trial {trial.number}] best val mAP@[.5:.95]={score:.4f} | {hp}")
    return score

In [ ]:
# ============================================================
# Run the RandomSearch study
# ============================================================
set_seed(SEED)

sampler = optuna.samplers.RandomSampler(seed=SEED)
study = optuna.create_study(direction="maximize", sampler=sampler)
study.optimize(objective, n_trials=MAX_TRIALS)

best_params = study.best_params
print("\n" + "=" * 60)
print("BEST TRIAL")
print("=" * 60)
print(f"best val mAP@[.5:.95]: {study.best_value:.4f}")
print("best params:", best_params)

# Persist the search results
os.makedirs(FINAL_DIR, exist_ok=True)
study.trials_dataframe().to_csv(os.path.join(FINAL_DIR, "hpo_trials.csv"), index=False)
with open(os.path.join(FINAL_DIR, "best_params.json"), "w") as f:
    json.dump({"best_value": study.best_value, "best_params": best_params}, f, indent=2)

In [ ]:
# ============================================================
# Final training with the best hyperparameters (full FINAL_EPOCHS)
# ============================================================
set_seed(SEED)

bp = study.best_params

model = RFDETRNano()
model.train(
    dataset_dir=DATASET_DIR,
    output_dir=FINAL_DIR,
    epochs=FINAL_EPOCHS,
    batch_size=bp.get("batch_size", 32),
    grad_accum_steps=1,
    resolution=FINAL_RESOLUTION,
    lr=bp["lr"],
    lr_encoder=bp["lr"] * bp["lr_encoder_mult"],
    weight_decay=bp["weight_decay"],
    lr_vit_layer_decay=bp["lr_vit_layer_decay"],
    drop_path=bp["drop_path"],
)

cleanup_gpu_memory(model)

In [ ]:
# ============================================================
# Reload the best checkpoint AT THE TRAINING RESOLUTION for inference
# (defaulting to 384 here would measure every metric at the wrong resolution)
# ============================================================
model = RFDETRNano(
    pretrain_weights=os.path.join(FINAL_DIR, "checkpoint_best_total.pth"),
    resolution=FINAL_RESOLUTION,
)
model.optimize_for_inference()

In [ ]:
# ============================================================
# Validation mAP (supervision)
# ============================================================
import supervision as sv
from tqdm import tqdm
from supervision.metrics import MeanAveragePrecision
from PIL import Image

val_ds = sv.DetectionDataset.from_coco(
    images_directory_path=f"{DATASET_DIR}/valid",
    annotations_path=f"{DATASET_DIR}/valid/_annotations.coco.json",
)

targets, predictions = [], []
for path, image, annotations in tqdm(val_ds, desc="Validation"):
    image = Image.open(path)
    detections = model.predict(image, threshold=0)  # threshold=0 is correct for mAP
    targets.append(annotations)
    predictions.append(detections)

val_map = MeanAveragePrecision().update(predictions, targets).compute()
print(val_map)

In [ ]:
# ============================================================
# Test-set mAP + inference speed with the tuned model
# ============================================================
import time

test_ds = sv.DetectionDataset.from_coco(
    images_directory_path=f"{DATASET_DIR}/test",
    annotations_path=f"{DATASET_DIR}/test/_annotations.coco.json",
)

test_targets, test_predictions = [], []
start = time.time()
for path, image, annotations in tqdm(test_ds, desc="Test"):
    image = Image.open(path)
    detections = model.predict(image, threshold=0.01)
    test_targets.append(annotations)
    test_predictions.append(detections)
elapsed = time.time() - start

test_map = MeanAveragePrecision().update(test_predictions, test_targets).compute()
n = len(test_ds)

report = f"""
========================================
    RF-DETR NANO (HPO) — TEST METRICS
========================================
Best HPO params:   {study.best_params}
Images:            {n}
Avg latency:       {elapsed / n * 1000:.2f} ms/image
Throughput:        {n / elapsed:.2f} img/s

--- Validation ---
{val_map}

--- Test ---
{test_map}
========================================
"""
print(report)

with open(os.path.join(FINAL_DIR, "comprehensive_metrics.txt"), "w") as f:
    f.write(report)

In [ ]:
# ============================================================
# Export the tuned model to ONNX (RF-DETR's built-in exporter)
# ============================================================
!pip install -q onnx onnxsim onnxruntime

In [ ]:
from rfdetr import RFDETRNano

# Reload the tuned checkpoint at the training resolution and use RF-DETR's own
# exporter, which traces the full detection pipeline (postprocessing included).
export_model = RFDETRNano(
    pretrain_weights=os.path.join(FINAL_DIR, "checkpoint_best_total.pth"),
    resolution=FINAL_RESOLUTION,
)
export_model.export(output_dir=os.path.join(FINAL_DIR, "onnx"), opset_version=17)
print("ONNX export written to", os.path.join(FINAL_DIR, "onnx"))

In [ ]:
# ============================================================
# Zip and download all HPO + evaluation artifacts
# ============================================================
!zip -r rfdetr_nano_hpo_results.zip /content/output

from google.colab import files
files.download("rfdetr_nano_hpo_results.zip")